# 06 — Baseline Models with Group-Split & IMR Controls

**Project:** Predicting Corporate GHG Intensity  
**Purpose:** Train and evaluate linear regressions (OLS, Ridge, Lasso) controlling for selection bias with the Probit-derived IMR.

---


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import FeatureEngineer
from src.models import ModelPipeline

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

### 1. Load Processed Features & Prepare Data

In [2]:
linked_df = pd.read_csv(INTERIM_DIR / "linked_panel.csv")

# Build leakage-free RAW features (ratios/target/year dummies only). Winsorization
# bounds, sector z-scores, and the Heckman Probit are fit per-split inside
# ModelPipeline.prepare_train_test(), using training rows only — never on test rows.
fe = FeatureEngineer()
raw_df = fe.build_raw_features(linked_df)

pipeline = ModelPipeline(target="scope1_intensity_rev", group_col="cik")
train_df, test_df, features = pipeline.prepare_train_test(raw_df)
print(f"Train observations: {len(train_df)}, Test observations: {len(test_df)}")
print("Modeling Features:", features)

2026-09-12 22:00:04,983 - INFO - Train years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)] (3803 raw rows), Test years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] (2675 raw rows)


2026-09-12 22:00:05,376 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4261, N = 3803, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 22:00:05,625 - INFO - Reporting subset: 1839 train / 1216 test firm-years, 28 features, target = scope1_intensity_rev


Train observations: 1839, Test observations: 1216
Modeling Features: ['size', 'leverage', 'roa', 'operating_margin', 'capex_intensity', 'rd_intensity', 'revenue_growth', 'inverse_mills_ratio', 'us_gdp_growth', 'us_co2_per_capita', 'us_energy_use_per_capita', 'year_2011', 'year_2012', 'year_2013', 'year_2014', 'year_2015', 'year_2016', 'year_2017', 'year_2018', 'year_2019', 'year_2020', 'year_2021', 'year_2022', 'year_2023', 'leverage_sector_z', 'roa_sector_z', 'operating_margin_sector_z', 'capex_intensity_sector_z']


### 2. Fit Baseline Linear Models
We fit OLS, Ridge, Lasso, and ElasticNet models.

In [3]:
results_df = pipeline.train_all_models(train_df, test_df, features, tune=False)
linear_results = results_df[results_df["model"].isin(["OLS", "Ridge", "Lasso", "ElasticNet"])]
linear_results[["model", "test_rmse", "test_r2", "test_mae"]]

E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
E:\Research_Projects\predicting-corporate-ghg-intensity\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


2026-09-12 22:00:12,931 - INFO - Trained 8 models.


,model,test_rmse,test_r2,test_mae
0,OLS,0.989321,-0.799550,0.211495
1,Ridge,0.971752,-0.736203,0.210264
2,Lasso,0.462388,0.606900,0.160162
3,ElasticNet,0.470730,0.592589,0.169044


### Discussion & Next Steps
Linear baselines provide a rigid benchmark. Next, we compare them with advanced non-linear ensembles (Random Forest, XGBoost, LightGBM) and deep PyTorch MLP regressors.